In [1]:
pip install torch torch_geometric scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.7 MB/s  0:00:00


In [2]:
import torch
from torch_geometric.datasets import Planetoid
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

In [7]:
# Loading the Cora dataset
# Overriding the URL directly to raw.githubusercontent.com to bypass redirection and header limit errors
from torch_geometric.datasets import Planetoid

Planetoid.url = 'https://raw.githubusercontent.com/kimiyoung/planetoid/master/data'
dataset = Planetoid(root='data/Planetoid', name='Cora')

Processing...
Done!


In [8]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

In [13]:
class CustomGNN(torch.nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super(CustomGNN, self).__init__()
    self.conv1 = GCNConv(input_dim, hidden_dim)
    self.conv2 = GCNConv(hidden_dim, output_dim)

  def forward(self, x, edge_index):
    # First GCN layer
    x = self.conv1(x, edge_index)
    x = F.relu(x)
    x = F.dropout(x, training=self.training)

    # Second GCN layer
    x = self.conv2(x, edge_index)
    return F.log_softmax(x, dim=1)

In [15]:
# Initialize the GNN model
input_features = dataset.num_features
hidden_features = 16
output_classes = dataset.num_classes

# Re-instantiating the model to apply the updated class definition
model = CustomGNN(input_features, hidden_features, output_classes)

In [16]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
graph_data = dataset[0]

def train_model():
  model.train()
  optimizer.zero_grad()
  # Pass the feature tensor directly because the forward method expects a tensor, not the data object
  output = model(graph_data.x, graph_data.edge_index)
  loss = F.nll_loss(output[graph_data.train_mask], graph_data.y[graph_data.train_mask])
  loss.backward()
  optimizer.step()
  return loss.item()

for epoch in range(200):
  loss = train_model()
  if (epoch + 1) % 10 == 0 or epoch == 0:
    print(f'Epoch: {epoch+1}, Loss: {loss:.4f}')

def evaluate_model(model, data):
  model.eval()
  with torch.no_grad():
    predictions = model(graph_data.x, graph_data.edge_index).argmax(dim=1)
    correct = predictions[graph_data.test_mask] == graph_data.y[graph_data.test_mask]
    accuracy = int(correct.sum()) / int(graph_data.test_mask.sum())
    return accuracy

accuracy = evaluate_model(model, graph_data)
print(f'Test Accuracy: {accuracy:.4f}')

Epoch: 1, Loss: 1.9503
Epoch: 10, Loss: 0.8337
Epoch: 20, Loss: 0.2956
Epoch: 30, Loss: 0.1217
Epoch: 40, Loss: 0.0577
Epoch: 50, Loss: 0.0472
Epoch: 60, Loss: 0.0434
Epoch: 70, Loss: 0.0730
Epoch: 80, Loss: 0.0589
Epoch: 90, Loss: 0.0461
Epoch: 100, Loss: 0.0314
Epoch: 110, Loss: 0.0363
Epoch: 120, Loss: 0.0496
Epoch: 130, Loss: 0.0435
Epoch: 140, Loss: 0.0375
Epoch: 150, Loss: 0.0443
Epoch: 160, Loss: 0.0275
Epoch: 170, Loss: 0.0262
Epoch: 180, Loss: 0.0288
Epoch: 190, Loss: 0.0177
Epoch: 200, Loss: 0.0222
Test Accuracy: 0.7990
